In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [4]:
import torch

from src.federated.client import (
    FederatedClient
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(device)

cuda


In [5]:
from src.preprocessing.dataset_loader import build_dataset_index
from src.preprocessing.splitter import create_stratified_split
from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)

In [20]:
train_paths, train_labels = build_dataset_index(
    "../data/raw/train"
)

test_paths, test_labels = build_dataset_index(
    "../data/raw/test"
)

(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [21]:
client1 = FederatedClient(
    "client_1",
    train_loader,
    device
)

client2 = FederatedClient(
    "client_2",
    train_loader,
    device
)

client3 = FederatedClient(
    "client_3",
    train_loader,
    device
)

In [22]:
client1.train(epochs=1)
print("--------------------")
client2.train(epochs=1)
print("--------------------")
client3.train(epochs=1)

0.0008577315020374954 0.9943912625312805
0.0006846990436315536 0.7692549228668213
2.003743611567188e-05 0.9998939037322998
0.0006220855284482241 0.7609496116638184
4.632762284018099e-05 0.9998699426651001
0.0013548305723816156 0.823218584060669
7.054402431094786e-07 0.9999966621398926
0.0006635702447965741 0.9155796766281128
4.3532540985324886e-06 0.9999896287918091
0.00048333348240703344 0.9511005282402039
1.150853859144263e-05 0.9999744892120361
0.001955424901098013 0.8751779794692993
3.994188773503993e-06 0.9999887943267822
0.0020584920421242714 0.9107434153556824
1.0396325706096832e-05 0.9999738931655884
0.0045571476221084595 0.9355827569961548
8.978237019618973e-06 0.9999797344207764
0.004542228765785694 0.9547286629676819
1.4609869140258525e-05 0.9999644756317139
0.005503748543560505 0.9519327282905579
2.283855792484246e-06 0.999993085861206
0.005050544161349535 0.9865124225616455
7.565562896161282e-07 0.9999974966049194
0.009448898024857044 0.9857316613197327
1.3294807104102802e

(0.37758800273633186, 0.8228371501272265)

In [23]:
weights = [

    client1.get_weights(),

    client2.get_weights(),

    client3.get_weights()

]


w1 = client1.get_weights()
w2 = client2.get_weights()

torch.equal(
    w1["classifier.2.weight"],
    w2["classifier.2.weight"]
)


False

In [24]:
from src.federated.fedAvg import (
    federated_average
)



global_weights = federated_average(
    weights
)

In [25]:
print(type(global_weights))

<class 'collections.OrderedDict'>


In [26]:
print(len(global_weights))

41


In [27]:
for key in list(global_weights.keys())[:10]:
    print(key)

features.0.weight
features.1.weight
features.1.bias
features.1.running_mean
features.1.running_var
features.1.num_batches_tracked
features.3.weight
features.4.weight
features.4.bias
features.4.running_mean


In [28]:
client1.set_weights(
    global_weights
)

client2.set_weights(
    global_weights
)

client3.set_weights(
    global_weights
)

In [29]:
w1 = client1.get_weights()
w2 = client2.get_weights()

In [30]:
torch.equal(
    w1["classifier.2.weight"],
    w2["classifier.2.weight"]
)

True

In [31]:
client1.set_weights(global_weights)
client2.set_weights(global_weights)

In [32]:
torch.equal(
    client1.get_weights()["classifier.2.weight"],
    client2.get_weights()["classifier.2.weight"]
)

True

In [33]:
all_equal = True

for key in client1.get_weights():

    if not torch.equal(
        client1.get_weights()[key],
        client2.get_weights()[key]
    ):
        all_equal = False
        break

print(all_equal)

True
